# Class-Conditioned Diffusion Utilities

The small conditional U-Net and DDPM helpers used to teach classifier-free guidance.

## Imports and defaults

Import shared numerical, model, image, and typing tools before defining reusable concepts.

In [ ]:
#| export
"""Class-conditioned U-Net and DDPM training utilities."""

from collections.abc import Sequence

import numpy as np
import torch
from numpy.typing import NDArray
from torch import Tensor, nn

DEFAULT_DEVICE: torch.device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
device: str = DEFAULT_DEVICE.type


## Conditioning embeddings

Map scalar times and one-hot classes into feature-wise vectors.

In [ ]:
#| export
class EmbedLayer(nn.Module):
    """Two-layer MLP for time or class-conditioning vectors."""

    def __init__(self, input_dim: int, emb_dim: int) -> None:
        """Initialize the embedding MLP."""
        super().__init__()
        self.input_dim = input_dim
        self.model = nn.Sequential(
            nn.Linear(input_dim, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
        )

    def forward(self, inputs: Tensor) -> Tensor:
        """Flatten inputs to ``(-1, input_dim)`` and embed them."""
        return self.model(inputs.view(-1, self.input_dim))


## U-Net convolution blocks

Build residual convolution, downsampling, and upsampling units.

In [ ]:
#| export
class ResidualConvBlock(nn.Module):
    """Two convolution blocks with an optional scaled residual path."""

    def __init__(
        self, in_channels: int, out_channels: int, is_res: bool = False
    ) -> None:
        """Initialize convolutions and residual behavior."""
        super().__init__()
        self.same_channels = in_channels == out_channels
        self.is_res = is_res
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, inputs: Tensor) -> Tensor:
        """Apply both convolutions and the configured residual path."""
        first: Tensor = self.conv1(inputs)
        second: Tensor = self.conv2(first)
        if not self.is_res:
            return second
        residual: Tensor = inputs if self.same_channels else first
        return (residual + second) / 1.414


class UnetDown(nn.Module):
    """Residual convolution followed by two-times spatial downsampling."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        """Initialize the downsampling block."""
        super().__init__()
        self.model = nn.Sequential(
            ResidualConvBlock(in_channels, out_channels), nn.MaxPool2d(2)
        )

    def forward(self, inputs: Tensor) -> Tensor:
        """Downsample a feature map."""
        return self.model(inputs)


class UnetUp(nn.Module):
    """Skip concatenation followed by transposed-convolution upsampling."""

    def __init__(self, in_channels: int, out_channels: int) -> None:
        """Initialize the upsampling block."""
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 2, 2),
            ResidualConvBlock(out_channels, out_channels),
            ResidualConvBlock(out_channels, out_channels),
        )

    def forward(self, inputs: Tensor, skip: Tensor) -> Tensor:
        """Concatenate a skip feature map and upsample."""
        return self.model(torch.cat((inputs, skip), dim=1))


## Conditional U-Net

Combine image features with masked class and timestep embeddings.

In [ ]:
#| export
class ConditionalUNet(nn.Module):
    """Small class-conditioned U-Net used for classifier-free guidance."""

    def __init__(
        self, in_channels: int, n_feat: int = 256, n_classes: int = 10
    ) -> None:
        """Initialize encoder, conditioning, and decoder blocks."""
        super().__init__()
        self.in_channels = in_channels
        self.n_feat = n_feat
        self.n_classes = n_classes
        self.init_conv = ResidualConvBlock(in_channels, n_feat, is_res=True)
        self.down1 = UnetDown(n_feat, n_feat)
        self.down2 = UnetDown(n_feat, 2 * n_feat)
        self.to_vec = nn.Sequential(nn.AvgPool2d(7), nn.GELU())
        self.timeembed1 = EmbedLayer(1, 2 * n_feat)
        self.timeembed2 = EmbedLayer(1, n_feat)
        self.contextembed1 = EmbedLayer(n_classes, 2 * n_feat)
        self.contextembed2 = EmbedLayer(n_classes, n_feat)
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(2 * n_feat, 2 * n_feat, 7, 7),
            nn.GroupNorm(8, 2 * n_feat),
            nn.ReLU(),
        )
        self.up1 = UnetUp(4 * n_feat, n_feat)
        self.up2 = UnetUp(2 * n_feat, n_feat)
        self.out = nn.Sequential(
            nn.Conv2d(2 * n_feat, n_feat, 3, 1, 1),
            nn.GroupNorm(8, n_feat),
            nn.ReLU(),
            nn.Conv2d(n_feat, in_channels, 3, 1, 1),
        )

    def forward(
        self,
        inputs: Tensor,
        classes: Tensor,
        timesteps: Tensor,
        context_mask: Tensor,
    ) -> Tensor:
        """Predict noise from images, classes, times, and context masks."""
        initial: Tensor = self.init_conv(inputs)
        down1: Tensor = self.down1(initial)
        down2: Tensor = self.down2(down1)
        hidden: Tensor = self.to_vec(down2)

        one_hot_classes: Tensor = nn.functional.one_hot(
            classes, num_classes=self.n_classes
        ).to(dtype=torch.float32)
        expanded_mask: Tensor = context_mask[:, None].repeat(1, self.n_classes)
        masked_classes: Tensor = one_hot_classes * (-(1 - expanded_mask))

        class_embedding1: Tensor = self.contextembed1(masked_classes).view(
            -1, self.n_feat * 2, 1, 1
        )
        time_embedding1: Tensor = self.timeembed1(timesteps).view(
            -1, self.n_feat * 2, 1, 1
        )
        class_embedding2: Tensor = self.contextembed2(masked_classes).view(
            -1, self.n_feat, 1, 1
        )
        time_embedding2: Tensor = self.timeembed2(timesteps).view(
            -1, self.n_feat, 1, 1
        )

        up1: Tensor = self.up0(hidden)
        up2: Tensor = self.up1(
            class_embedding1 * up1 + time_embedding1, down2
        )
        up3: Tensor = self.up2(
            class_embedding2 * up2 + time_embedding2, down1
        )
        return self.out(torch.cat((up3, initial), dim=1))


# Backward-compatible class name from the upstream notebooks.
Unet = ConditionalUNet


## Forward diffusion schedule

Precompute coefficients for adding noise and reversing the process.

In [ ]:
#| export
def noise_scheduler(num_timesteps: int) -> dict[str, Tensor]:
    """Precompute linear DDPM noise-schedule coefficients.

    Args:
        num_timesteps: Number of forward diffusion steps.

    Returns:
        Named one-dimensional schedule tensors of length
        ``num_timesteps + 1``.
    """
    beta_start, beta_end = 0.0001, 0.02
    beta: Tensor = (
        (beta_end - beta_start)
        * torch.arange(0, num_timesteps + 1, dtype=torch.float32)
        / num_timesteps
        + beta_start
    )
    alpha: Tensor = 1 - beta
    alpha_bar: Tensor = torch.cumsum(torch.log(alpha), dim=0).exp()
    sqrt_one_minus_alpha_bar: Tensor = torch.sqrt(1 - alpha_bar)
    return {
        "alpha_t": alpha,
        "oneover_sqrta": 1 / torch.sqrt(alpha),
        "sqrt_beta_t": torch.sqrt(beta),
        "alphabar_t": alpha_bar,
        "sqrtab": torch.sqrt(alpha_bar),
        "sqrtmab": sqrt_one_minus_alpha_bar,
        "mab_over_sqrtmab": (1 - alpha) / sqrt_one_minus_alpha_bar,
    }


## DDPM training wrapper

Sample noisy states and optimize direct noise prediction.

In [ ]:
#| export
class DDPM(nn.Module):
    """Training wrapper for class-conditioned denoising diffusion."""

    def __init__(
        self,
        model: nn.Module,
        n_T: int,
        device: torch.device | str = DEFAULT_DEVICE,
        drop_prob: float = 0.1,
    ) -> None:
        """Initialize the model, schedule buffers, and MSE objective."""
        super().__init__()
        self.model = model.to(device)
        for name, values in noise_scheduler(n_T).items():
            self.register_buffer(name, values)
        self.n_T = n_T
        self.device = torch.device(device)
        self.drop_prob = drop_prob
        self.loss_mse = nn.MSELoss()

    def forward(self, images: Tensor, classes: Tensor) -> Tensor:
        """Sample noisy training states and return noise-prediction loss."""
        timesteps: Tensor = torch.randint(
            1, self.n_T + 1, (images.shape[0],), device=self.device
        )
        noise: Tensor = torch.randn_like(images)
        noisy_images: Tensor = (
            self.sqrtab[timesteps, None, None, None] * images
            + self.sqrtmab[timesteps, None, None, None] * noise
        )
        context_mask: Tensor = torch.bernoulli(
            torch.zeros_like(classes) + self.drop_prob
        ).to(self.device)
        predicted_noise: Tensor = self.model(
            noisy_images, classes, timesteps / self.n_T, context_mask
        )
        return self.loss_mse(noise, predicted_noise)


## Classifier-free guided sampling

Combine conditional and unconditional predictions while stepping backward through noise.

In [ ]:
#| export
@torch.no_grad()
def sample(
    ddpm: DDPM,
    model: nn.Module,
    n_sample: int,
    size: Sequence[int],
    device: torch.device | str,
    guide_w: float = 0.0,
    step_size: int = 1,
) -> tuple[Tensor, NDArray[np.floating]]:
    """Sample images with classifier-free guidance.

    Args:
        ddpm: DDPM wrapper containing schedule buffers.
        model: Conditional noise-prediction model.
        n_sample: Number of images to generate. The upstream class schedule
            expects a multiple of ten.
        size: Per-image ``(channels, height, width)`` shape.
        device: Sampling device.
        guide_w: Classifier-free guidance strength.
        step_size: Reverse-process timestep stride.

    Returns:
        Final image tensor and selected intermediate NumPy snapshots.
    """
    current: Tensor = torch.randn(n_sample, *size, device=device)
    classes: Tensor = torch.arange(0, 10, device=device)
    classes = classes.repeat(int(n_sample / classes.shape[0]))
    context_mask: Tensor = torch.zeros_like(classes, device=device)
    classes = classes.repeat(2)
    context_mask = context_mask.repeat(2)
    context_mask[n_sample:] = 1.0

    snapshots: list[NDArray[np.floating]] = []
    for timestep in range(ddpm.n_T, 0, -step_size):
        times: Tensor = torch.full(
            (n_sample, 1, 1, 1),
            timestep / ddpm.n_T,
            device=device,
        )
        doubled_images: Tensor = current.repeat(2, 1, 1, 1)
        doubled_times: Tensor = times.repeat(2, 1, 1, 1)
        noise: Tensor = (
            torch.randn(n_sample, *size, device=device)
            if timestep > 1
            else torch.zeros_like(current)
        )
        predictions: Tensor = model(
            doubled_images, classes, doubled_times, context_mask
        )
        conditional = predictions[:n_sample]
        unconditional = predictions[n_sample:]
        guided: Tensor = (1 + guide_w) * conditional - guide_w * unconditional
        current = (
            ddpm.oneover_sqrta[timestep]
            * (current - guided * ddpm.mab_over_sqrtmab[timestep])
            + ddpm.sqrt_beta_t[timestep] * noise
        )
        if timestep % 20 == 0 or timestep == ddpm.n_T or timestep < 8:
            snapshots.append(current.detach().cpu().numpy())
    return current, np.array(snapshots)


## Summary

The module exposes typed U-Net blocks, a device-aware DDPM wrapper, and classifier-free guided sampling.